# Practical Work: YAML → TSV with Age Calculation

This notebook generates a dataset, saves it as YAML, then reads it, computes age, and exports to TSV.

In [ ]:
import yaml
import random
from datetime import datetime, timedelta
from faker import Faker
import csv
import os
from tqdm import tqdm

TOTAL = 10000
BATCH_SIZE = 100
DATA_PATH = "../data/1/"

In [14]:
def random_date(start_year=1970, end_year=2023):
    start = datetime(start_year, 1, 1)
    end = datetime(end_year, 12, 31)
    return start + timedelta(days=random.randint(0, (end - start).days))

In [15]:
def calculate_age(birth_date_str):
    birth_date = datetime.strptime(birth_date_str, "%Y-%m-%d")
    today = datetime.today()
    age = today.year - birth_date.year

    if (today.month, today.day) < (birth_date.month, birth_date.day):
        age -= 1

    return age

In [17]:
fake = Faker()

def generate_user(user_id):
    name = fake.name()
    emails = [fake.email() for _ in range(random.randint(1, 3))]

    registration_date = random_date(2015, 2023)
    last_login = registration_date + timedelta(days=random.randint(0, 1000))

    publications = []
    for _ in range(random.randint(1, 3)):
        reviews = []
        for _ in range(random.randint(0, 3)):
            reviews.append({
                "user_id": random.randint(1, 1000),
                "text": fake.text(max_nb_chars=100),
            })

        publications.append({
            "title": fake.sentence(),
            "description": fake.text(max_nb_chars=200),
            "pages": random.randint(1, 50),
            "category": random.choice(["tech", "science", "art", "news"]),
            "date": random_date(2018, 2024).strftime("%Y-%m-%d"),
            "reviews": reviews
        })

    return {
        "id": user_id,
        "name": name,
        "emails": emails,
        "registration_date": registration_date.strftime("%Y-%m-%d"),
        "last_login": last_login.strftime("%Y-%m-%d"),
        "is_verified": random.choice([True, False]),
        "birth_date": random_date(1970, 2010).strftime("%Y-%m-%d"),
        "gender": random.choice(["male", "female"]),
        "publications": publications
    }

In [18]:
os.makedirs(DATA_PATH, exist_ok=True)

with open(DATA_PATH + "input.yaml", "w", encoding="utf-8") as f:

    for start in tqdm(range(1, TOTAL + 1, BATCH_SIZE), desc="Generating users"):

        batch = [
            generate_user(i)
            for i in range(start, min(start + BATCH_SIZE, TOTAL + 1))
        ]

        for user in batch:
            yaml.dump(user, f, allow_unicode=True, sort_keys=False)
            f.write("---\n")

Generating users: 100%|██████████| 10/10 [00:03<00:00,  2.80it/s]


In [19]:
with open(DATA_PATH + "input.yaml", "r", encoding="utf-8") as f:
    first_user = next(yaml.safe_load_all(f))

print(yaml.dump(first_user, allow_unicode=True, sort_keys=False))

id: 1
name: Angela George
emails:
- xthomas@example.net
registration_date: '2022-11-23'
last_login: '2024-05-29'
is_verified: true
birth_date: '1999-03-14'
gender: male
publications:
- title: Order position stuff this.
  description: 'Marriage stop charge there development government. Staff a factor
    term race run threat hair.

    Interview movie itself effect analysis whose to. Bad back play still herself.'
  pages: 48
  category: science
  date: '2019-04-25'
  reviews:
  - user_id: 47
    text: Test big rise energy morning fear. Ahead raise new firm. Report kitchen
      think until interview.
- title: Number better up risk window author box.
  description: 'Much evening form.

    Billion Republican account campaign only kitchen. Coach person party theory. Republican
    federal wear imagine those station respond.

    Health prove realize those budget seek.'
  pages: 45
  category: science
  date: '2020-09-14'
  reviews:
  - user_id: 115
    text: Tree sign huge huge teacher se

In [20]:
def stream_users(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        for user in yaml.safe_load_all(f):
            if user is not None:
                yield user

In [21]:
def stream_batches(generator, batch_size=BATCH_SIZE):
    batch = []

    for item in generator:
        batch.append(item)

        if len(batch) == batch_size:
            yield batch
            batch = []

    if batch:
        yield batch

In [24]:
with open(DATA_PATH + "output.tsv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f, delimiter="\t")

    writer.writerow([
        "id", "name", "age", "emails_count",
        "registration_date", "last_login", "verified"
    ])

    user_stream = stream_users(DATA_PATH + "input.yaml")

    with tqdm(total=TOTAL, desc="Processing users") as pbar:
        for batch in stream_batches(user_stream, BATCH_SIZE):
            for user in batch:
                writer.writerow([
                    user["id"],
                    user["name"],
                    calculate_age(user["birth_date"]),
                    len(user["emails"]),
                    user["registration_date"],
                    user["last_login"],
                    user["is_verified"]
                ])
                pbar.update(1)

Processing users: 100%|██████████| 1000/1000 [00:03<00:00, 328.26it/s]


In [25]:
with open(DATA_PATH + "output.tsv", "r", encoding="utf-8") as f:
    for i in range(5):
        print(f.readline().strip())

id	name	age	emails_count	registration_date	last_login	verified
1	Angela George	27	1	2022-11-23	2024-05-29	True
2	Darlene Hartman	39	2	2022-12-16	2024-09-21	True
3	Taylor Bailey	29	1	2016-10-01	2018-10-20	True
4	Jennifer Lang	47	3	2021-04-30	2023-10-30	True
